# The Importance of Baseline Models & Traditional ML

---

### Why Start with a Baseline Model?

* **Start Cheap and Simple:** Simple heuristics and traditional models require minimal compute, are fast to iterate on, and incur negligible infrastructure costs.
* **Establish a Performance Benchmark:** Creates a measurable lower bound ($R^2$, MSE, MAE) to determine whether complex models or LLMs actually provide meaningful lift.
* **Prevent Over-Engineering:** Demonstrates when traditional algorithms suffice, avoiding the latency, financial cost, and unpredictability of Large Language Models when an LLM is not the right tool for the job.

---

### Machine Learning Overview

* **Definition:** A branch of Artificial Intelligence where statistical algorithms learn patterns directly from data to make predictions rather than following hardcoded, rule-based instructions.
* **Traditional vs. Modern ML:** In modern workflows, the term "Machine Learning" often refers to classical algorithms (e.g., Linear Regression, Decision Trees, Random Forests, XGBoost) developed prior to Deep Learning and LLMs.
* **Relevance for LLM Practitioners:** Understanding foundational ML principles—such as loss functions, data leakage, feature engineering, and evaluation splits—remains essential for designing and evaluating LLM-driven applications.

---

### Core Concepts

* **Generalization:** The capability of a trained model to make accurate predictions on new, unseen data by learning underlying patterns rather than memorizing examples.
* **Overfitting:** Occurs when a model memorizes the training data—including noise, outliers, and artifacts—causing it to fail when evaluating unseen test data.

> **Primary Objective:** The ultimate goal of any machine learning system is **Generalization**—producing reliable, accurate outputs on data it has never encountered before.

# Price Prediction Pipeline: Data Loading, Baselines & Feature Engineering

This script sets up a foundation for an e-commerce price prediction system by loading data from the Hugging Face Hub, establishing naive statistical baselines, and engineering structured tabular features.

---

### Key Workflow Stages

* **Environment & Data Ingestion**:
* Configures dataset mode via `LITE_MODE` (`ujjwalsingh108/items_lite` vs. `items_full`).
* Loads split partitions (`train`, `val`, `test`) into custom `Item` data objects using `Item.from_hub()`.
* Logs partition counts to verify sample sizes across splits.


* **Heuristic & Statistical Baselines**:
* **Random Baseline (`random_pricer`)**: Predicts a random integer between $1$ and $999$ to establish a worst-case benchmark evaluated on the `val` split.
* **Mean Constant Baseline (`constant_pricer`)**: Calculates the global arithmetic mean ($\mu$) from `training_prices` and predicts that constant value across the unseen `test` split.


* **Feature Engineering & Tabular Structuring**:
* **`get_features(item)`**: Extracts model input features per item:
* `weight`: Continuous item weight.
* `weight_unknown`: Binary indicator flag ($1$ if weight is $0$, else $0$).
* `text_length`: Character length of `item.summary` (with a safe fallback to `""` for missing/`None` values).


* **`list_to_dataframe(items)`**: Converts lists of `Item` objects into clean Pandas DataFrames (`train_df`, `test_df`) with aligned input feature columns and the target variable column (`df['price']`).



---

### Pipeline Summary Table

| Pipeline Component | Implementation | Input | Output / Role |
| --- | --- | --- | --- |
| **Data Loader** | `Item.from_hub(dataset)` | Hugging Face repo path | `train`, `val`, `test` collections |
| **Baseline 1** | `random_pricer` | `Item` | Random uniform integer ($1$–$999$) |
| **Baseline 2** | `constant_pricer` | `Item` | Global training average price ($\mu$) |
| **Feature Extractor** | `get_features` | Single `Item` | Feature dictionary (`weight`, `text_length`, etc.) |
| **Dataset Vectorizer** | `list_to_dataframe` | `List[Item]` | Structured `pd.DataFrame` ($X$ features + $y$ target) |

In [26]:
import random
import pandas as pd
import numpy as np
from rich import print
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.ensemble import RandomForestRegressor
from price_agent.data.evaluator import evaluate
from price_agent.data.items import Item


In [3]:
LITE_MODE = True  # Set to False to use the full dataset

In [27]:
username = "ujjwalsingh108"
dataset = f"{username}/items_lite" if LITE_MODE else f"{username}/items_full"

train, val, test = Item.from_hub(dataset)

print(f"Loaded {len(train):,} training items, {len(val):,} validation items, {len(test):,} test items")

2026-08-16 23:50:35,127 INFO httpx HTTP Request: HEAD https://huggingface.co/datasets/ujjwalsingh108/items_lite/resolve/main/README.md "HTTP/1.1 307 Temporary Redirect"
2026-08-16 23:50:35,257 INFO httpx HTTP Request: HEAD https://huggingface.co/api/resolve-cache/datasets/ujjwalsingh108/items_lite/457bd961c240529a681fe965e44297f6d98a19a8/README.md "HTTP/1.1 200 OK"
2026-08-16 23:50:35,568 INFO httpx HTTP Request: HEAD https://huggingface.co/datasets/ujjwalsingh108/items_lite/resolve/457bd961c240529a681fe965e44297f6d98a19a8/items_lite.py "HTTP/1.1 404 Not Found"
2026-08-16 23:50:36,575 INFO httpx HTTP Request: HEAD https://s3.amazonaws.com/datasets.huggingface.co/datasets/datasets/ujjwalsingh108/items_lite/ujjwalsingh108/items_lite.py "HTTP/1.1 404 Not Found"
2026-08-16 23:50:36,889 INFO httpx HTTP Request: HEAD https://huggingface.co/datasets/ujjwalsingh108/items_lite/resolve/457bd961c240529a681fe965e44297f6d98a19a8/.huggingface.yaml "HTTP/1.1 404 Not Found"
2026-08-16 23:50:37,298 INF

Loaded 20,000 training items, 1,000 validation items, 1,000 test items

In [6]:
def random_pricer(item:Item)->float:
    return random.randrange(1,1000)

In [28]:
# Evaluate baseline performance on validation data
eval_result = evaluate(random_pricer, val)
print(eval_result)

  0%|          | 0/200 [00:00<?, ?it/s]

$218 $750 $702 $571 $848 $166 $873 $372 $431 $493 $243 $354 $799 $287 $309 $866 $286 $3 $559 $68 $120 $499 $970 $490 $184 $113 $485 $359 $7 $713 $72 $134 $11 $136 $563 $29 $395 $753 $76 $235 $505 $350 $575 $21 $105 $173 $320 $916 $90 $4 $920 $697 $139 $83 $598 $648 $502 $43 $678 $188 $201 $274 $237 $834 $686 $265 $109 $662 $495 $186 $706 $827 $310 $649 $627 $869 $297 $346 $372 $538 $909 $14 $106 $127 $24 $450 $833 $185 $63 $738 $435 $500 $871 $46 $670 $419 $541 $856 $507 $581 $254 $627 $636 $382 $308 $611 $511 $469 $352 $547 $538 $638 $392 $65 $75 $398 $775 $585 $394 $199 $311 $835 $78 $425 $616 $206 $18 $885 $166 $635 $552 $188 $516 $353 $440 $255 $494 $853 $140 $201 $595 $362 $703 $516 $632 $490 $81 $588 $205 $808 $912 $589 $698 $289 $372 $532 $690 $688 $187 $111 $407 $393 $105 $35 $154 $728 $198 $300 $77 $726 $801 $70 $746 $136 $378 $743 $534 $158 $98 $168 $844 $589 $750 $792 $838 $602 $862 $137 $303 $244 $844 $265 $236 $373 $774 $235 $121 $427 $275 $379 

None

In [29]:
# That was fun!
# We can do better - here's another rather trivial model

training_prices = [item.price for item in train]
training_average = sum(training_prices) / len(training_prices)
print(training_average)

def constant_pricer(item:Item)->float:
    return training_average

67.02510799999999

In [30]:
eval_result = evaluate(constant_pricer, test) # running on unseen test data
print(eval_result)

  0%|          | 0/200 [00:00<?, ?it/s]

$78 $41 $51 $41 $27 $121 $20 $32 $56 $208 $332 $265 $48 $31 $673 $52 $1 $22 $48 $27 $46 $133 $103 $37 $17 $48 $43 $38 $12 $61 $28 $52 $38 $51 $13 $225 $2 $31 $66 $52 $47 $65 $49 $55 $3 $54 $54 $51 $31 $18 $53 $37 $167 $21 $41 $37 $60 $55 $51 $23 $40 $12 $0 $47 $78 $32 $51 $162 $31 $54 $21 $55 $48 $40 $47 $54 $22 $59 $53 $40 $7 $44 $59 $24 $34 $53 $57 $823 $0 $39 $51 $50 $48 $37 $31 $46 $51 $51 $54 $58 $60 $52 $53 $3 $46 $54 $52 $43 $36 $117 $51 $40 $38 $48 $38 $33 $45 $41 $23 $28 $57 $41 $51 $55 $14 $50 $47 $31 $104 $27 $40 $17 $53 $9 $53 $32 $39 $27 $33 $25 $34 $31 $15 $55 $55 $50 $39 $48 $53 $57 $55 $89 $40 $93 $21 $38 $37 $3 $13 $47 $322 $54 $58 $54 $683 $46 $57 $53 $42 $54 $57 $11 $277 $28 $51 $303 $77 $23 $45 $50 $133 $30 $233 $55 $22 $45 $30 $7 $8 $27 $223 $50 $57 $24 $41 $21 $152 $32 $55 $27 

None

* **Target vs. Feature Separation:** In machine learning, price is the target variable ($y$) you want to predict, while weight, weight_unknown, and text_length are the input features ($X$).
---
* **Inference Reusability (Data Leakage):** When using the model in production or inside the evaluate loop (predictor(item)), you only have access to the item's metadata—the actual item.price is what the model needs to guess. If get_features(item) includes price, your feature extraction pipeline will fail or cause target leakage when predicting unseen data.
---
* **Single Responsibility Principle:** get_features should strictly extract model inputs ($X$). Setting df['price'] separately allows you to split the data cleanly into X = train_df.drop(columns=['price']) and y = train_df['price'].

In [14]:
def get_features(item:Item)->dict:
    return {
        "weight": item.weight,
        "weight_unknown": 1 if item.weight==0 else 0,
        "text_length": len(item.summary or "")
    }

`list_to_dataframe(items: List[Item]) -> pd.DataFrame:`

`features = [get_features(item) for item in items]:` Applies `get_features` to each Item in the list via list comprehension.

`df = pd.DataFrame(features):` Converts the list of feature dictionaries into a tabular DataFrame where dictionary keys become column headers.

`df['price'] = [item.price for item in items]:` Extracts the ground truth price for each item and appends it as the target column.

In [15]:
from typing import List

def list_to_dataframe(items: List[Item]) -> pd.DataFrame:
    features = [get_features(item) for item in items]
    df = pd.DataFrame(features)
    df['price'] = [item.price for item in items]
    return df

train_df = list_to_dataframe(train)
test_df = list_to_dataframe(test)

# Traditional Linear Regression Workflow

This script demonstrates training, inspecting, and evaluating a multivariate **Ordinary Least Squares (OLS) Linear Regression** model using structured tabular features.

---

### 1. Data Preparation & Matrix Splitting

```python
np.random.seed(42)

# Define feature subset and separate predictors (X) from the target (y)
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

```

* **`np.random.seed(42)`**: Fixes the random seed to ensure deterministic, reproducible results across executions.
* **Feature Separation**: Splits the tabular datasets into input feature matrices ($X$) and target vectors ($y$) for both the training and test splits.

---

### 2. Model Fitting

```python
model = LinearRegression()
model.fit(X_train, y_train)

```

The algorithm estimates the optimal weights ($\beta$) that minimize the residual sum of squares between observed prices and predicted values:

$$\hat{y} = \beta_0 + \beta_1(\text{weight}) + \beta_2(\text{weight\_unknown}) + \beta_3(\text{text\_length})$$

---

### 3. Model Inspection & Interpretation

```python
for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

```

* **`model.coef_` ($\beta_1, \beta_2, \beta_3$)**: Represents the marginal change in predicted price for a one-unit change in that feature (holding all other variables constant).
* **Positive coefficient**: Increases predicted price as the feature value rises.
* **Negative coefficient**: Decreases predicted price as the feature value rises.


* **`model.intercept_` ($\beta_0$)**: The expected baseline price when all input feature values are zero.

---

### 4. Prediction & Performance Evaluation

```python
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

```

* **`model.predict(X_test)`**: Computes continuous price predictions ($\hat{y}$) for all instances in the unseen test set.
* **`mean_squared_error` (MSE)**: Measures the average squared difference between predictions and actual prices:

$$\text{MSE} = \frac{1}{n} \sum_{i=1}^{n} (y_i - \hat{y}_i)^2$$


* **`r2_score` ($R^2$)**: Quantifies the proportion of target variance explained by the features:

$$R^2 = 1 - \frac{\sum (y_i - \hat{y}_i)^2}{\sum (y_i - \bar{y})^2}$$


* $R^2 \approx 0$: Performs no better than a trivial mean baseline.
* $R^2 > 0$: Outperforms simple average guessing (closer to $1.0$ represents a superior linear fit).



---

### 5. Integration with Evaluator Pipeline

To pass the trained linear regression model to the custom Plotly visualization suite in `evaluator.py`:

```python
def linear_pricer(item: Item) -> float:
    """Wrapper function to evaluate individual Item instances."""
    features_df = pd.DataFrame([get_features(item)])[feature_columns]
    return float(model.predict(features_df)[0])


# Run interactive visual evaluation
evaluate(linear_pricer, test)

```

In [31]:
# Traditional Linear Regression!

np.random.seed(42)

# Separate features and target
feature_columns = ['weight', 'weight_unknown', 'text_length']

X_train = train_df[feature_columns]
y_train = train_df['price']
X_test = test_df[feature_columns]
y_test = test_df['price']

# Train a Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)

for feature, coef in zip(feature_columns, model.coef_):
    print(f"{feature}: {coef}")
print(f"Intercept: {model.intercept_}")

# Predict the test set and evaluate
y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print(f"Mean Squared Error: {mse}")
print(f"R-squared Score: {r2}")

weight: 3.8737376494319893

weight_unknown: 3.5272855935003955

text_length: 0.0945908553941823

Intercept: 26.562640844275023

Mean Squared Error: 11494.403698713106

R-squared Score: 0.10814210052147144

In [17]:
def linear_pricer(item: Item) -> float:
    # 1. Extract features into a 1-row DataFrame
    item_features = pd.DataFrame([get_features(item)])[feature_columns]
    # 2. Predict and return float
    return float(model.predict(item_features)[0])

In [33]:
# Evaluate with interactive charts
evaluate(linear_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$22 $33 $53 $35 $26 $85 $26 $77 $54 $214 $278 $273 $38 $28 $660 $40 $13 $25 $41 $22 $43 $57 $104 $29 $8 $52 $44 $39 $17 $53 $41 $45 $34 $48 $20 $194 $7 $16 $73 $50 $54 $56 $49 $51 $3 $58 $47 $47 $27 $19 $50 $26 $174 $14 $34 $28 $53 $69 $47 $18 $41 $11 $12 $46 $76 $28 $43 $157 $19 $43 $12 $41 $48 $41 $48 $47 $19 $53 $46 $36 $22 $39 $67 $29 $33 $51 $43 $819 $1 $33 $43 $42 $42 $25 $19 $28 $37 $52 $51 $56 $46 $42 $39 $11 $39 $47 $58 $54 $24 $121 $43 $37 $31 $41 $48 $28 $37 $33 $33 $23 $51 $43 $44 $50 $21 $54 $48 $25 $91 $22 $39 $10 $44 $11 $55 $27 $43 $22 $32 $30 $32 $23 $14 $56 $49 $47 $40 $42 $43 $47 $43 $30 $36 $84 $14 $41 $33 $3 $6 $34 $191 $45 $53 $50 $661 $50 $43 $47 $39 $54 $56 $0 $90 $23 $44 $119 $71 $22 $44 $40 $131 $20 $28 $47 $26 $40 $30 $5 $7 $21 $226 $39 $51 $19 $45 $34 $156 $18 $58 $23 

# Natural Language Linear Regression Pricing Model

This workflow builds an NLP-driven regression pipeline that converts unstructured product summaries into numerical text vectors and trains a **Linear Regression** model directly on word occurrences to predict prices.

---

### 1. Data Extraction & Formatting

```python
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

```

* **`prices`**: Extracts the ground-truth product prices from the training split and converts them into a 1D NumPy array of floats to act as the target variable ($y$).
* **`documents`**: Collects the raw text summary from each training item into a list of strings to serve as the raw input corpus.

---

### 2. Bag-of-Words Text Vectorization

```python
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words="english")
X = vectorizer.fit_transform(documents)

```

* **`CountVectorizer`**: Implements a Bag-of-Words (BoW) model that tokenizes text, builds a vocabulary, and tracks word frequency counts per item.
* **`max_features=2000`**: Restricts the vocabulary to the 2,000 most frequent words across all documents, capping dimensionality and filtering rare noise words.
* **`stop_words='english'`**: Drops standard, non-informative English words (such as *"the"*, *"and"*, *"in"*, *"is"*).
* **`fit_transform(documents)`**:
* Learns the vocabulary mapping from `documents` (**fit**).
* Converts the text corpus into a sparse numerical matrix $X$ of shape `(n_items, 2000)` (**transform**), where each cell represents word counts.



---

### 3. Model Training

```python
regressor = LinearRegression()
regressor.fit(X, prices)

```

* Trains a standard Ordinary Least Squares (OLS) Linear Regression model directly on the sparse text matrix $X$.
* Learns a regression coefficient (price weight) for each of the 2,000 vocabulary words (e.g., words like *"leather"* or *"4K"* learn positive price contributions, while generic terms learn neutral or lower weights).

---

### 4. Single-Item Inference & Post-Processing

```python
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

```

* **`vectorizer.transform([item.summary])`**: Transforms the incoming item's summary text using the **pre-fitted vocabulary** (without refitting). Unseen words are automatically ignored.
* **`regressor.predict(x)[0]`**: Computes the linear dot-product price prediction from word counts.
* **`max(..., 0)` (Non-Negative Flooring)**: Standard linear models can predict negative numbers for short or heavily-penalized text; the `max(..., 0)` clamp guarantees predicted prices never drop below $0.00.

---

### 5. Evaluation

```python
evaluate(natural_language_linear_regression_pricer, test)

```

* Passes the callable prediction function to `evaluator.py`.
* Concurrently evaluates predictions on the `test` split, reporting Mean Absolute Error, MSE, $R^2$, and rendering interactive Plotly scatter & error trend visual charts.

In [20]:
prices = np.array([float(item.price) for item in train])
documents = [item.summary for item in train]

In [21]:
np.random.seed(42)
vectorizer = CountVectorizer(max_features=2000, stop_words='english')
X = vectorizer.fit_transform(documents)

In [34]:
# Here are the 1,000 most common words that it picked, not including "stop words":

selected_words = vectorizer.get_feature_names_out()
print(f"Number of selected words: {len(selected_words)}")
print("Selected words:", selected_words[1000:1020])

Number of selected words: 2000

Selected words: ['jigsaw' 'joint' 'joints' 'jump' 'kastar' 'keeping' 'keeps' 'key'
 'keyboard' 'keychain' 'keychains' 'keys' 'kg' 'khz' 'kicker' 'kickstand'
 'kids' 'kit' 'kitchen' 'kits']

In [35]:
regressor = LinearRegression()
regressor.fit(X, prices)

,"fit_intercept fit_intercept: bool, default=TrueWhether to calculate the intercept for this model. If setto False, no intercept will be used in calculations(i.e. data is expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"tol tol: float, default=1e-6The precision of the solution (`coef_`) is determined by `tol` whichspecifies the convergence criterion of the underlying solver. `tol` isset as `atol` and `btol` of :func:`scipy.sparse.linalg.lsqr` whenfitting on sparse training data. `tol` is set as `cond` of:func:`scipy.linalg.lstsq` when fitting on dense training data... versionadded:: 1.7.. versionchanged:: 1.9 Now supported on dense data, interpreted as the `cond` parameter.",1e-06
,"n_jobs n_jobs: int, default=NoneThe number of jobs to use for the computation. This will only providespeedup in case of sufficiently large problems, that is if firstly`n_targets > 1` and secondly `X` is sparse or if `positive` is setto `True`. ``None`` means 1 unless in a:obj:`joblib.parallel_backend` context. ``-1`` means using allprocessors. See :term:`Glossary <n_jobs>` for more details.",None
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive. Thisoption is only supported for dense arrays.For a comparison between a linear regression model with positive constraintson the regression coefficients and a linear regression without such constraints,see :ref:`sphx_glr_auto_examples_linear_model_plot_nnls.py`... versionadded:: 0.24",False
Name,Type,Value
"coef_ coef_: array of shape (n_features, ) or (n_targets, n_features)Estimated coefficients for the linear regression problem.If multiple targets are passed during the fit (y 2D), thisis a 2D array of shape (n_targets, n_features), while if onlyone target is passed, this is a 1D array of length n_features.","ndarray[float64](2000,)","[-2.04, 1.45,-1. ,..., 4.71,-8.19,44.82]"
"intercept_ intercept_: float or array of shape (n_targets,)Independent term in the linear model. Set to 0.0 if`fit_intercept = False`.",float64,193.1
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,2000


In [36]:
def natural_language_linear_regression_pricer(item):
    x = vectorizer.transform([item.summary])
    return max(regressor.predict(x)[0], 0)

In [37]:
evaluate(natural_language_linear_regression_pricer, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$128 $4 $16 $3 $10 $110 $12 $35 $11 $228 $39 $244 $55 $18 $84 $15 $35 $102 $12 $30 $21 $10 $78 $26 $20 $19 $29 $3 $62 $6 $26 $23 $40 $130 $71 $137 $25 $21 $6 $7 $15 $11 $35 $156 $6 $2 $13 $5 $20 $41 $23 $11 $121 $46 $19 $30 $2 $12 $15 $42 $44 $1 $43 $12 $75 $35 $17 $54 $4 $1 $11 $40 $19 $25 $20 $27 $45 $33 $69 $27 $23 $18 $38 $5 $1 $14 $10 $339 $36 $44 $3 $31 $3 $8 $36 $31 $70 $11 $10 $33 $76 $8 $35 $74 $14 $42 $15 $56 $24 $42 $16 $80 $8 $59 $57 $33 $80 $16 $14 $49 $2 $73 $6 $12 $14 $4 $19 $9 $86 $110 $12 $2 $70 $25 $18 $10 $18 $16 $19 $16 $33 $4 $5 $22 $11 $17 $89 $12 $24 $10 $12 $151 $30 $68 $46 $29 $24 $1 $2 $10 $252 $4 $9 $6 $131 $55 $53 $14 $228 $76 $10 $17 $279 $20 $31 $205 $122 $14 $43 $5 $17 $3 $60 $2 $61 $1 $12 $42 $5 $6 $8 $57 $20 $31 $160 $78 $178 $14 $11 $68 

# Random Forest Regression Model

The **Random Forest** is an **ensemble** algorithm that combines multiple smaller models to produce more accurate, robust, and generalizable predictions.

---

### How Decision Trees Work

At its foundation, Random Forest relies on a simpler machine learning model called a **decision tree**. A decision tree functions like a hierarchical flowchart composed of sequential `IF/THEN` conditional rules applied to input features:

* In our model, each **feature** corresponds to a word count token in the Bag-of-Words vector (the number of times a specific word appears in the product summary).
* A single tree's logic works conceptually like this:

```text
Decision Tree Logic:
└── IF "TV" count > 3
    └── IF "LED" count > 2
        └── IF "HD" count >= 1
            └── Predicted Price = $500

```

> **The Limitation:** While individual decision trees are fast and easy to interpret, they are prone to **overfitting**—memorizing the exact noise of the training data rather than generalizing well to unseen samples.

---

### The Power of the Ensemble Forest

A Random Forest resolves individual decision tree overfitting by constructing an ensemble of many trees (defaulting to **100 trees**):

1. **Bootstrap Data Sampling:** Each tree trains on a random subset of the training dataset (sampled with replacement).
2. **Feature Subsetting:** At every split within a tree, the algorithm considers only a random subset of all vocabulary features (words).
3. **Ensemble Averaging:** To produce the final prediction for a continuous target (price), the model calculates the arithmetic mean of the predictions from all 100 individual trees.

---

### Single-Item Inference & Evaluation

To evaluate the Random Forest model on the test dataset:

```python
def random_forest(item):
    """Vectorizes an item's summary and returns a non-negative Random Forest prediction."""
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])


# Run visual and statistical evaluation on test data
evaluate(random_forest, test)

```

* **`vectorizer.transform([item.summary])`**: Extracts word frequency counts using the existing fitted vocabulary.
* **`rf_model.predict(x)[0]`**: Aggregates predictions across the 100 decision trees.
* **`max(0, ...)`**: Clamps the prediction so that estimated prices never fall below $0.00.

---

### Saving the Trained Model

Because training large ensembles on high-dimensional text matrices can be computationally intensive, serialize the fitted model to disk for fast reuse using `joblib`:

```python
import joblib

# Persist the trained model to disk
joblib.dump(rf_model, "random_forest.joblib")

# To reload the model later:
# loaded_rf = joblib.load("random_forest.joblib")

```

In [38]:
subset = 15_000
rf_model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=4)
rf_model.fit(X[:subset], prices[:subset])

,"n_jobs n_jobs: int, default=NoneThe number of jobs to run in parallel. :meth:`fit`, :meth:`predict`,:meth:`decision_path` and :meth:`apply` are all parallelized over thetrees. ``None`` means 1 unless in a :obj:`joblib.parallel_backend`context. ``-1`` means using all processors. See :term:`Glossary<n_jobs>` for more details.",4
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"ma

In [39]:
def random_forest(item):
    x = vectorizer.transform([item.summary])
    return max(0, rf_model.predict(x)[0])

In [40]:
evaluate(random_forest, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$54 $9 $9 $7 $3 $154 $9 $5 $26 $230 $144 $301 $12 $7 $18 $3 $4 $3 $26 $13 $5 $54 $20 $3 $22 $6 $55 $23 $45 $16 $24 $2 $65 $34 $1 $223 $23 $13 $32 $11 $5 $67 $34 $64 $18 $19 $14 $4 $11 $57 $22 $4 $194 $29 $11 $14 $18 $10 $22 $0 $11 $19 $6 $53 $71 $5 $8 $57 $1 $11 $17 $20 $1 $13 $1 $10 $61 $7 $2 $0 $8 $1 $47 $5 $1 $15 $2 $275 $10 $4 $2 $20 $26 $20 $19 $37 $22 $11 $16 $34 $15 $0 $23 $128 $3 $9 $1 $71 $17 $137 $20 $66 $5 $24 $39 $2 $12 $2 $42 $46 $4 $47 $0 $7 $8 $16 $12 $9 $79 $94 $8 $14 $23 $12 $4 $5 $6 $15 $18 $25 $7 $14 $175 $27 $4 $4 $41 $2 $1 $10 $2 $53 $4 $64 $22 $2 $12 $12 $19 $8 $317 $15 $7 $13 $172 $7 $17 $19 $92 $23 $6 $43 $273 $14 $6 $317 $110 $7 $23 $0 $123 $8 $159 $30 $20 $2 $33 $12 $5 $8 $73 $14 $6 $42 $47 $60 $191 $15 $6 $19 

# Introducing XGBoost Regression

**XGBoost** (Extreme Gradient Boosting) is a tree-based ensemble algorithm designed for computational efficiency, speed, and predictive performance.

---

### Key Concepts: Random Forest vs. XGBoost

Both models are tree ensembles, but they construct and combine their trees differently:

* **Random Forest (Bagging):** Builds many deep, independent trees in parallel from random subsets of the data and features, then averages their predictions.
* **XGBoost (Boosting):** Builds trees sequentially (one after another). Each subsequent tree is trained to predict and correct the **residuals (errors)** made by all previous trees using **gradient descent**.
* **Performance & Speed:** XGBoost incorporates internal tree pruning, regularization, and hardware acceleration, allowing it to train quickly on full datasets with strong generalization.

---

### Implementation & Code Breakdown

```python
import xgboost as xgb

np.random.seed(42)

# Initialize the gradient boosted trees regressor
xgb_model = xgb.XGBRegressor(
    n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1
)

# Train on the full Bag-of-Words text feature matrix
xgb_model.fit(X, prices)

```

* **`n_estimators=1000`**: Constructs a sequence of up to 1,000 boosting rounds (trees).
* **`learning_rate=0.1` ($\eta$)**: Scales the contribution of each new tree to prevent individual trees from dominating the model and causing overfitting.
* **`n_jobs=4`**: Distributes tree construction across 4 CPU cores in parallel.
* **`xgb_model.fit(X, prices)`**: Fits the boosted model across the full word-count matrix `X` and target price array `prices`.

---

### Single-Item Inference & Evaluation

```python
def xg_boost(item: Item) -> float:
    """Transforms an item's summary text and generates an XGBoost price prediction."""
    x = vectorizer.transform([item.summary])
    return max(0, float(xgb_model.predict(x)[0]))


# Run visual and statistical evaluation on test data
evaluate(xg_boost, test)

```

* **`vectorizer.transform([item.summary])`**: Maps the item summary to the existing 2,000-word vocabulary without refitting.
* **`xgb_model.predict(x)[0]`**: Passes the sparse vector through the additive tree sequence.
* **`max(0, ...)`**: Clamps the output at zero to ensure non-negative price predictions.

---

### Environment Setup Note

If `import xgboost` raises an `OpenMP` library loading error on macOS, install the required dependency via Homebrew in your terminal:

```bash
brew install libomp

```

In [42]:
import xgboost as xgb

np.random.seed(42)

# Initialize the gradient boosted trees regressor
xgb_model = xgb.XGBRegressor(
    n_estimators=1000, random_state=42, n_jobs=4, learning_rate=0.1
)

# Train on the full Bag-of-Words text feature matrix
xgb_model.fit(X, prices)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,True
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",None
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [43]:
def xg_boost(item: Item) -> float:
    """Transforms an item's summary text and generates an XGBoost price prediction."""
    x = vectorizer.transform([item.summary])
    return max(0, float(xgb_model.predict(x)[0]))


# Run visual and statistical evaluation on test data
evaluate(xg_boost, test)

  0%|          | 0/200 [00:00<?, ?it/s]

$31 $17 $13 $22 $12 $140 $14 $27 $0 $220 $96 $283 $23 $11 $59 $3 $7 $5 $38 $3 $14 $68 $11 $16 $23 $3 $61 $9 $40 $21 $42 $3 $49 $131 $64 $207 $25 $3 $78 $6 $20 $64 $39 $42 $2 $6 $30 $5 $7 $68 $16 $18 $163 $28 $7 $4 $26 $5 $16 $15 $16 $21 $33 $12 $68 $2 $15 $146 $15 $27 $4 $44 $3 $18 $1 $16 $10 $2 $4 $18 $11 $34 $50 $23 $23 $7 $2 $460 $17 $9 $12 $18 $26 $28 $7 $43 $56 $8 $8 $90 $28 $2 $26 $134 $8 $4 $1 $53 $18 $97 $15 $58 $6 $81 $23 $1 $37 $7 $31 $69 $3 $53 $3 $6 $24 $19 $16 $2 $106 $108 $1 $4 $53 $10 $1 $11 $5 $16 $51 $22 $12 $18 $220 $33 $16 $8 $42 $3 $2 $5 $1 $144 $3 $57 $13 $1 $3 $3 $8 $22 $317 $24 $9 $11 $179 $18 $17 $4 $243 $33 $2 $39 $227 $3 $23 $274 $115 $41 $24 $2 $11 $15 $78 $19 $20 $5 $51 $13 $34 $13 $68 $13 $7 $78 $76 $103 $251 $5 $4 $47 